## Loading Model and Input Files 

In [5]:
# 🔄 CLEAN INITIALIZATION - RESTART KERNEL IF NEEDED
print("🔄 Initializing environment...")
import pathlib
from google import genai
from google.genai import types
import os
import asyncio
import json
import fitz  # PyMuPDF for PDF processing
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if API key is loaded
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment variables. Please check your .env file.")

# Configure the API
client = genai.Client(api_key=api_key)


pa_file = "PA.pdf"
referral_file = "referral_package.pdf"

# Retrieve and encode the PDF byte
filepath = pathlib.Path('Input Data/Adbulla/' + pa_file)

print("✅ Setup complete!")


🔄 Initializing environment...
✅ Setup complete!



## 📄 Referral Package Analysis & Information Extraction to Json Using Gemeni 

In [6]:
# 🔄 DIRECT GEMINI OCR & ANALYSIS - Bypassing Mistral
print("🔄 Using Gemini directly for OCR and information extraction...")

# Enhanced prompt for direct PDF analysis
referral_prompt = """You are analyzing a referral package PDF (collection of scanned medical documents) to extract patient information that will be used to fill a Prior Authorization form.

CONTEXT: This referral package contains scanned documents like insurance cards, medical history notes, test results, and other supporting documentation. You need to perform OCR on the PDF and extract structured information.

EXTRACTION FOCUS: Use your vision capabilities to read and extract all available information and display it in order as it is read from the PDF. Then turn that output into JSON format:

PATIENT DEMOGRAPHICS:
- Full name, DOB, gender, address, phone numbers
- Patient ID numbers, MRN, account numbers
- Emergency contacts and relationships

INSURANCE INFORMATION:
- Insurance company name and plan details
- Member ID, group number, policy number
- Subscriber information and relationship to patient
- Coverage details and effective dates

CLINICAL INFORMATION:
- Primary and secondary diagnoses with ICD codes
- Current medications, dosages, and frequencies
- Allergies and adverse reactions
- Vital signs and lab results
- Previous treatments and outcomes
- Medical history and comorbidities

PROVIDER INFORMATION:
- Referring physician name, specialty, and contact info
- Practice name and address
- NPI numbers and license information
- Facility details where treatment will occur

TREATMENT DETAILS:
- Requested medication/procedure/device
- Dosage, frequency, duration
- Medical necessity justification
- Previous treatment failures
- Clinical criteria met for approval

SUPPORTING EVIDENCE:
- Lab results supporting diagnosis
- Imaging studies and results
- Functional assessments
- Specialist consultations
- Treatment response documentation

INSTRUCTIONS:
1. Carefully read through all pages of the PDF
2. Extract text and data from tables, forms, and documents
3. Pay attention to medical terminology and abbreviations
4. Look for provider contact information sections
5. Identify medication/product information from clinical notes
6. First return all info read in the document. List all information from tables including reference ranges and other details. Do not skip anything that has information
7. Then return a structured JSON with all relevant extracted information, using null for missing data"""

# Load and process the referral package PDF directly with Gemini
referral_pdf_path = 'Input Data/Adbulla/referral_package.pdf'

# Upload the PDF file to Gemini using new API
print(f"📄 Loading referral package: {referral_pdf_path}")
referral_file = client.files.upload(file=referral_pdf_path)
print(f"✅ File uploaded successfully: {referral_file.name}")

# Wait for file processing
import time
while referral_file.state == "PROCESSING":
    print("⏳ Processing file...")
    time.sleep(2)
    referral_file = client.files.get(name=referral_file.name)

if referral_file.state == "FAILED":
    raise ValueError("❌ File processing failed")
print("✅ File processed successfully")

# Extract structured data from referral package using Gemini vision
referral_response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=[referral_prompt, referral_file]
)
referral_data = referral_response.text
print("📊 Referral Package Data Extracted:")
print(referral_data)

# Store the extracted text for compatibility with existing code
referral_package_text = referral_data

🔄 Using Gemini directly for OCR and information extraction...
📄 Loading referral package: Input Data/Adbulla/referral_package.pdf
✅ File uploaded successfully: files/b5molqlrwisu
✅ File processed successfully
📊 Referral Package Data Extracted:
Here's the extracted information in two formats as requested:

---

## 1. All Information Read from the PDF (In Order)

**Page 1**
*   **Header:** 04/22/2024 WED 11:32 FAX 2001/028
*   **Sender Information (Top Left):** BetterLife
*   **FAX TRANSMISSION Header:**
    *   Better Life Multiple Sclerosis Center
    *   3320 Montgomery Dr. Nashville, TN 37361
    *   F: 615-562-4820 P: 615-562-4848
    *   Dr. Asriel Han | Dr. Aditya Shah
*   **TO:** Golden Gate Infusion Center
*   **Fax:** 614-278-3355
*   **Phone:** 614895-7655
*   **From:** Erfan Rostami, BSN, RN
*   **P:** 615-343-1176
*   **F:** 615-343-1219
*   **Pages (including cover sheet):** (The '1' appears to be part of the header '2001/028' indicating page 1 of 28, not a page count here.

## PA widget input info extraction

In [7]:
import fitz

# Code from Headstarter tutorial to extract fields from PDF
def extracting_fields_from_pdf(pdf_file):
    """
    Extracts form fields from a PDF file and returns them as a list of dictionaries.
    Each dictionary contains field name, type, value, page number, and other metadata.
    """
    if not pathlib.Path(pdf_file).exists():
        raise FileNotFoundError(f"The file {pdf_file} does not exist.")
    doc = fitz.open(pdf_file)
    fields = []
    for page_num, page in enumerate(doc, start = 1):
        for w in page.widgets() or []:
            field = {
                "name": w.field_name,
                "type": "checkbox" if w.field_type == fitz.PDF_WIDGET_TYPE_CHECKBOX else "text",
                "value": w.field_value,
                "page": page_num,
                "field_type": w.field_type,
                "field_type_string": w.field_type_string,
                "field_label": w.field_label,
            }
            print(field)
            fields.append(field)

    field_by_page = {}
    for field in fields:
        page_num = field["page"]
        if page_num not in field_by_page:
            field_by_page[page_num] = []
    
    return field_by_page

pa_fields = extracting_fields_from_pdf("Input Data/Adbulla/PA.pdf")


{'name': 'CB1', 'type': 'checkbox', 'value': '', 'page': 2, 'field_type': 2, 'field_type_string': 'CheckBox', 'field_label': 'Start of treatment'}
{'name': 'T2', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Start date: (MM)'}
{'name': 'T3', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Start date: (DD)'}
{'name': 'T4', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Start date: (YYYY)'}
{'name': 'CB5', 'type': 'checkbox', 'value': '', 'page': 2, 'field_type': 2, 'field_type_string': 'CheckBox', 'field_label': 'Continuation of therapy'}
{'name': 'T6', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Date of last treatment: (MM)'}
{'name': 'T7', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Date of last treatment: (D

## PA feild organization Prompt template for Gemeni 

In [8]:
def format_fields_for_pa(pa_fields):
  return """
  You are given an array of dictionaries, each representing a form field extracted from a PDF. Each dictionary contains:
  - "name": the field name
  - "type": the field type (e.g., "text", "checkbox")
  - "value": the field value (may be null)
  - "page": the 1-based page number
  - "field_type": the PDF widget type code
  - "field_type_string": the PDF widget type as a string
  - "field_label": the field label (may be null)

  TASK:
  1. Organize these fields into a hierarchical JSON tree grouped by page number.
  2. For each page, create an array of fields, preserving all metadata.
  3. The top-level JSON should be:
  {
    "pages": {
      "1": [ ...fields... ],
      "2": [ ...fields... ],
      ...
    }
  }
  4. Do not omit any fields or metadata.
  5. Return only valid JSON (no markdown, no extra commentary).

  Example output:
  {
    "pages": {
      "1": [
        {
          "name": "T1",
          "type": "text",
          "value": "",
          "page": 1,
          "field_type": 2,
          "field_type_string": "Text",
          "field_label": "First Name"
        },
        ...
      ],
      "2": [
        ...
      ]
    }
  }

  <PA_FIELDS>
  {pa_fields}
  </PA_FIELDS> """

## Send Prompt to Gemeni 

In [9]:
# Send to Gemini using NEW 2025 SDK

async def query_gemini_async(prompt, pdf_path, model_name='gemini-2.5-flash'):
    """
    Async function to query Gemini with a PDF and prompt using the new 2025 Google Gen AI SDK
    """
    filepath = pathlib.Path(pdf_path)
    
    # Create PDF part using new SDK structure
    pdf_bytes = filepath.read_bytes()
    pdf_part = types.Part.from_bytes(
        data=pdf_bytes, 
        mime_type="application/pdf"
    )
    
    # Use async client from new SDK
    response = await client.aio.models.generate_content(
        model=model_name,
        contents=[pdf_part, prompt]
    )
    
    print(response.text)
    return response.text

In [ ]:
pa_fields_context = {}

# Process each page of the PA form
async def process_page(page):
    prompt = format_fields_for_pa(pa_fields[page])
    print (prompt)

    result = await query_gemini_async(prompt, "Input Data/Adbulla/PA.pdf")
    print ("---------------------------------")
    return page, result
# create a list of tasks to run in parallel
tasks = [process_page(page) for page in pa_fields]

# running the tasks in parallel
results = await asyncio.gather(*tasks)

# 
for page, result in results:
    pa_fields_context[page] = result



  You are given an array of dictionaries, each representing a form field extracted from a PDF. Each dictionary contains:
  - "name": the field name
  - "type": the field type (e.g., "text", "checkbox")
  - "value": the field value (may be null)
  - "page": the 1-based page number
  - "field_type": the PDF widget type code
  - "field_type_string": the PDF widget type as a string
  - "field_label": the field label (may be null)

  TASK:
  1. Organize these fields into a hierarchical JSON tree grouped by page number.
  2. For each page, create an array of fields, preserving all metadata.
  3. The top-level JSON should be:
  {
    "pages": {
      "1": [ ...fields... ],
      "2": [ ...fields... ],
      ...
    }
  }
  4. Do not omit any fields or metadata.
  5. Return only valid JSON (no markdown, no extra commentary).

  Example output:
  {
    "pages": {
      "1": [
        {
          "name": "T1",
          "type": "text",
          "value": "",
          "page": 1,
          "fiel

In [7]:
# Create PA_form_analysis from pa_fields_context
# This combines all the processed PA field data into a single analysis

if 'pa_fields_context' in locals() and pa_fields_context:
    # Combine all page results into a single analysis
    PA_form_analysis = ""
    for page, result in pa_fields_context.items():
        PA_form_analysis += f"PAGE {page}:\n{result}\n\n"
    print("✅ PA_form_analysis created successfully!")
    print(f"Length: {len(PA_form_analysis)} characters")
else:
    # Fallback: create from raw pa_fields if pa_fields_context isn't ready
    PA_form_analysis = json.dumps(pa_fields, indent=2)
    print("⚠️  Using fallback: PA_form_analysis created from raw pa_fields")
    print("Note: Run cell 12 first to get processed analysis")

print("PA_form_analysis preview:")
print(PA_form_analysis)


✅ PA_form_analysis created successfully!
Length: 257801 characters
PA_form_analysis preview:
PAGE 2:
```json
{
  "pages": {
    "1": [
      {
        "name": "Phone_1",
        "type": "text",
        "value": "1-866-503-0857",
        "page": 1,
        "field_type": 2,
        "field_type_string": "Text",
        "field_label": null
      },
      {
        "name": "Fax_1",
        "type": "text",
        "value": "1-844-268-7263",
        "page": 1,
        "field_type": 2,
        "field_type_string": "Text",
        "field_label": null
      },
      {
        "name": "Availity_1",
        "type": "text",
        "value": "https://www.aetna.com/health-care-professionals/resource-center/availity.html",
        "page": 1,
        "field_type": 2,
        "field_type_string": "Text",
        "field_label": null
      },
      {
        "name": "Phone_2",
        "type": "text",
        "value": "1-855-463-0933",
        "page": 1,
        "field_type": 2,
        "field_type_string"

In [8]:
pa_fields_context

{2: '```json\n{\n  "pages": {\n    "1": [\n      {\n        "name": "Phone_1",\n        "type": "text",\n        "value": "1-866-503-0857",\n        "page": 1,\n        "field_type": 2,\n        "field_type_string": "Text",\n        "field_label": null\n      },\n      {\n        "name": "Fax_1",\n        "type": "text",\n        "value": "1-844-268-7263",\n        "page": 1,\n        "field_type": 2,\n        "field_type_string": "Text",\n        "field_label": null\n      },\n      {\n        "name": "Availity_1",\n        "type": "text",\n        "value": "https://www.aetna.com/health-care-professionals/resource-center/availity.html",\n        "page": 1,\n        "field_type": 2,\n        "field_type_string": "Text",\n        "field_label": null\n      },\n      {\n        "name": "Phone_2",\n        "type": "text",\n        "value": "1-855-463-0933",\n        "page": 1,\n        "field_type": 2,\n        "field_type_string": "Text",\n        "field_label": null\n      },\n      {\n

## form filling strategy 

In [ ]:
# 🤖 INTELLIGENT AUTO-MAPPING: Let AI map actual field names to patient data
print("🤖 Creating intelligent automated field mapping...")

# Get all actual PDF fields with their labels
doc = fitz.open("Input Data/Adbulla/PA.pdf")
pdf_fields = []
for page_num in range(len(doc)):
    page = doc[page_num]
    for widget in page.widgets():
        if widget.field_name and widget.field_label:
            pdf_fields.append({
                'field_name': widget.field_name,
                'field_type': widget.field_type_string,
                'field_label': widget.field_label,
                'page': page_num + 1
            })
doc.close()

    # Fill PDF with AI mappings - Enhanced with debugging
def fill_pdf_with_ai_mappings(input_pdf_path, output_pdf_path, mappings):
    doc = fitz.open(input_pdf_path)
    filled_count = 0
    skipped_count = 0
    
    print(f"🔧 Starting PDF filling with {len(mappings)} mappings...")
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        
        for widget in page.widgets():
            field_name = widget.field_name
            
            if field_name in mappings:
                value = mappings[field_name]
                
                # Skip empty values
                if value == "" or value is None:
                    skipped_count += 1
                    continue
                
                try:
                    if widget.field_type == fitz.PDF_WIDGET_TYPE_CHECKBOX:
                        widget.field_value = bool(value)
                    elif widget.field_type == fitz.PDF_WIDGET_TYPE_TEXT:
                        widget.field_value = str(value)
                    else:
                        print(f"⚠️  Unknown field type for {field_name}: {widget.field_type}")
                        continue
                    
                    widget.update()
                    filled_count += 1
                    print(f"✅ {field_name}: {value}")
                    
                except Exception as e:
                    print(f"⚠️  Error filling {field_name}: {e}")
            else:
                # Debug: Show unmapped fields (first 10 only)
                if filled_count + skipped_count < 10:
                    print(f"🔍 Field '{field_name}' not in mappings")
    
    print(f"📊 Summary: {filled_count} fields filled, {skipped_count} skipped (empty values)")
    doc.save(output_pdf_path)
    doc.close()
    return filled_count


🤖 Creating intelligent automated field mapping...


In [ ]:
# 🚀 IMPROVED INTELLIGENT AUTO-MAPPING with Enhanced Coverage
print("🚀 Creating ENHANCED intelligent automated field mapping...")

# Enhanced mapping prompt that addresses all the gaps
enhanced_mapping_prompt = f"""You are an expert medical form automation specialist. Create comprehensive JSON mapping for a Prior Authorization form.

Pay attention to the whole context of the form and the patient data. And pay attention to this whole prompt including the middle parts

CRITICAL REQUIREMENTS:
1. Fill ALL possible fields where patient data is available
2. Handle checkbox dependencies (if main checkbox is checked, fill related sub-fields)
3. Include product/treatment information from clinical data
4. Use medical terminology appropriately
5. Format dates correctly for each field type
6. Handle multiple choice selections intelligently
7. NEVER use placeholder text like "(From Clinical Data)" - use actual data or leave blank

PDF FIELDS AVAILABLE:
{json.dumps(pdf_fields[:80], indent=2)}

PATIENT DATA AVAILABLE:
{referral_data}

SPECIFIC MAPPING INSTRUCTIONS:

A. PATIENT INFORMATION SECTION:
- Map full name to separate first/last name fields
- Map complete address (street, city, state, zip)
- Map all phone numbers (home, work, cell)
- Map DOB in correct format
- Map patient demographics

B. INSURANCE INFORMATION:
- Map member ID, group number, insured name
- Check for secondary coverage and map if available
- Map insurance carrier information

C. PRESCRIBER INFORMATION - CRITICAL RULES:
- Look for "Provider Contact Information" or similar headings in the clinical data
- Search for doctor/physician names in message records or clinical notes for inference if no difinitive contact information is found
- Extract actual prescriber contact details (address, phone, fax)
- Look for NPI numbers, DEA numbers in provider sections
- Check credentials (M.D., D.O., N.P., P.A.) from provider information
- If prescriber information is not clearly available, LEAVE FIELDS BLANK
- DO NOT use placeholder text like "Prescriber Address (From Clinical Data)"

D. TREATMENT/PRODUCT INFORMATION - ENHANCED SEARCH:
- Look for headings containing "Product", "Medication", "Drug", "Treatment"
- Search doctor comments and clinical notes for prescribed medications
- Look for drug names, dosages, frequencies in clinical documentation
- Check for diagnosis codes (ICD-10) in clinical data
- Search for treatment justification in physician notes
- If specific product/medication is not found, LEAVE FIELDS BLANK

E. DISPENSING/ADMINISTRATION:
- Determine administration method from clinical data
- Map pharmacy/dispensing location if specified
- Map administration codes if available

F. CHECKBOX LOGIC:
- If "Start of treatment" is checked, fill start date fields
- If "Continuation of therapy" is checked, fill last treatment date
- For dispensing location, check appropriate option based on data
- For prescriber type, check M.D., D.O., N.P., or P.A. based on credentials

G. DATE FORMATTING:
- For separate MM/DD/YYYY fields, split dates accordingly
- For single date fields, use MM/DD/YYYY format
- Extract dates from clinical timeline

H. MISSING DATA HANDLING - STRICT RULES:
- Only map fields where you have ACTUAL, SPECIFIC subject data
- If information is not clearly available in the clinical data, LEAVE THE FIELD BLANK
- Do not guess, weakly infer, or fabricate information
- Do not use placeholder text or generic descriptions
- Better to have fewer accurate fields than many inaccurate ones

SEARCH STRATEGY FOR PRESCRIBER INFO:
1. Search for "Provider Contact", "Physician Contact", "Doctor Information"
2. Look in message records for sender/physician information
3. Check clinical notes for prescribing physician details
4. If no specific prescriber info found, leave prescriber fields blank

SEARCH STRATEGY FOR PRODUCT INFO:
1. Look for explicit "Product" or "Medication" sections
2. Search doctor comments for drug names and prescriptions
3. Check treatment plans for specific medications
4. Look for pharmaceutical names in clinical documentation
5. If no specific product found, leave product fields blank

OUTPUT FORMAT - CRITICAL JSON REQUIREMENTS:
1. Return ONLY valid JSON (no markdown blocks, no extra text)
2. Ensure all JSON brackets and braces are properly closed
3. Use double quotes for all strings
4. Boolean values must be true/false (not True/False)
5. Do not truncate the JSON response

{{
  "field_mappings": {{
    "T12": "Shakh",
    "T13": "Abdulla", 
    "CB1": true,
    "T2": "05",
    "T3": "22", 
    "T4": "2024",
    "CB5": false,
    "T14": "04/01/2001",
    "T15": "425 Sherman Ave APT D",
    "T16": "Nashville",
    "T17": "TN",
    "T18": "37995"
  }},
  "mapping_notes": {{
    "total_fields_mapped": 0,
    "sections_completed": [],
    "missing_data_areas": [],
    "checkbox_decisions": {{}}
  }}
}}

IMPORTANT: Complete the entire JSON response. Do not truncate or cut off the JSON in the middle.

ANALYZE THE CLINICAL DATA THOROUGHLY and create comprehensive mappings using ONLY actual data found."""

# Send enhanced prompt to Gemini
print("🧠 Asking AI for ENHANCED comprehensive field mappings...")
# Extract structured data from referral package using Gemini vision
enhanced_response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=[enhanced_mapping_prompt]
)


# Extract and apply enhanced mappings with ROBUST error handling
def robust_json_parser(json_text):
    """Advanced JSON parser that handles common formatting issues and truncation"""
    import re
    
    # Step 1: Extract JSON content
    if "```json" in json_text:
        json_start = json_text.find("```json") + 7
        json_end = json_text.find("```", json_start)
        if json_end == -1:  # No closing ```
            json_content = json_text[json_start:].strip()
        else:
            json_content = json_text[json_start:json_end].strip()
    else:
        json_content = json_text.strip()
    
    # Step 2: Try direct parsing first
    try:
        return json.loads(json_content)
    except json.JSONDecodeError:
        pass
    
    # Step 3: Advanced cleaning and repair
    print("🔧 Applying advanced JSON repair...")
    
    # Fix common issues
    json_content = re.sub(r'"CB\d+": tr(?![ue])', lambda m: m.group(0).replace('tr', 'true'), json_content)
    json_content = re.sub(r'"CB\d+": fa(?![lse])', lambda m: m.group(0).replace('fa', 'false'), json_content)
    
    # Step 4: Extract field mappings manually using regex
    field_mappings = {}
    
    # Pattern for text fields: "T12": "value"
    text_pattern = r'"(T\d+(?:\.\d+)?|CB\d+|Provider Admin [^"]+)"\s*:\s*"([^"]*)"'
    for match in re.finditer(text_pattern, json_content):
        field_mappings[match.group(1)] = match.group(2)
    
    # Pattern for boolean fields: "CB1": true/false
    bool_pattern = r'"(T\d+(?:\.\d+)?|CB\d+|Provider Admin [^"]+)"\s*:\s*(true|false)'
    for match in re.finditer(bool_pattern, json_content):
        field_mappings[match.group(1)] = match.group(2) == 'true'
    
    # Pattern for null fields: "T1": null
    null_pattern = r'"(T\d+(?:\.\d+)?|CB\d+|Provider Admin [^"]+)"\s*:\s*null'
    for match in re.finditer(null_pattern, json_content):
        field_mappings[match.group(1)] = None
    
    if field_mappings:
        print(f"✅ Extracted {len(field_mappings)} field mappings using regex parsing")
        return {"field_mappings": field_mappings, "mapping_notes": {"extraction_method": "regex"}}
    
    # Step 5: Last resort - create minimal mapping from visible data
    print("⚠️  Using fallback minimal mapping")
    return {
        "field_mappings": {
            "T12": "Shakh",
            "T13": "Abdulla", 
            "T14": "04/01/2001",
            "T15": "425 Sherman Aves APT D",
            "T16": "Nashville",
            "T17": "TN",
            "T18": "37923"
        },
        "mapping_notes": {"extraction_method": "fallback"}
    }

try:
    # Use the robust parser
    enhanced_mappings = robust_json_parser(enhanced_response.text)
    
    enhanced_field_mappings = enhanced_mappings.get("field_mappings", {})
    mapping_notes = enhanced_mappings.get("mapping_notes", {})
    
    print(f"\n🎯 ENHANCED AI created {len(enhanced_field_mappings)} comprehensive field mappings")
    
    # Show mapping summary
    if mapping_notes:
        print(f"📊 Total fields mapped: {mapping_notes.get('total_fields_mapped', len(enhanced_field_mappings))}")
        print(f"📋 Sections completed: {', '.join(mapping_notes.get('sections_completed', []))}")
        if mapping_notes.get('missing_data_areas'):
            print(f"⚠️  Missing data areas: {', '.join(mapping_notes.get('missing_data_areas', []))}")
    
    # Show sample enhanced mappings
    print("\nSample enhanced mappings:")
    for i, (field, value) in enumerate(list(enhanced_field_mappings.items())[:8]):
        print(f"  • {field} → {value}")
    
    # Apply enhanced mappings to PDF
    output_pdf_enhanced = "Input Data/Adbulla/PA_FILLED_ENHANCED.pdf"
    filled_count = fill_pdf_with_ai_mappings("Input Data/Adbulla/PA.pdf", output_pdf_enhanced, enhanced_field_mappings)
    
    print(f"\n🎉 ENHANCED MAPPING SUCCESS!")
    print(f"✅ Filled {filled_count} fields using enhanced AI mapping")
    print(f"📄 Enhanced PDF saved as: {output_pdf_enhanced}")
    
    # Compare with previous version
    previous_count = len(field_mappings) if 'field_mappings' in locals() else 0
    improvement = filled_count - previous_count
    if improvement > 0:
        print(f"🚀 IMPROVEMENT: +{improvement} more fields filled than previous version!")
    
except Exception as e:
    print(f"❌ Error in enhanced mapping: {e}")
    print("Full enhanced response:")
    print(enhanced_response.text)


🚀 Creating ENHANCED intelligent automated field mapping...
🧠 Asking AI for ENHANCED comprehensive field mappings...
✅ Enhanced AI mapping completed!
Response preview:
```json
{
  "field_mappings": {
    "CB1": true,
    "T2": "05",
    "T3": "22",
    "T4": "2024",
    "CB5": false,
    "T6": "",
    "T7": "",
    "T8": "",
    "T9": "Erfan Rostami, BSN, RN",
    "T10": "615-343-1176",
    "T11": "615-343-1219",
    "T12": "Shakh",
    "T13": "Abdulla",
    "T14": "04/01/2001",
    "T15": "425 Sherman Ave APT D",
    "T16": "Nashville",
    "T17": "TN",
    "T18": "37923",
    "T19": "865-395-3958",
    "T20": "",
    "T21": "865-395-0481",
    "T22": "",
    "T24": "",
    "T25": "",
    "T26": "",
    "T27": "",
    "T23": "No Known Allergies",
    "T28": "LAJM14345116",
    "T29": "435000",
    "T30": "ABDULLA, SHAKH",
    "CB31": false,
    "CB32": true,
    "T33": "",
    "T34": "",
    "T35": "",
    "T42": "Hao H",
    "T43": "Gu",
    "CB44": tr...

🎯 ENHANCED AI created 80 com